# Identity Studio + Fooocus (All-in-One)

## Головне — два різні інструменти

| Лінк | Для чого |
|------|----------|
| **Fooocus** (7865) | **Тут генеруєш картинки** (промпт, Ragnarok, без цензури) |
| **Identity Studio** (7866) | **Опційно** — підставити обличчя з фото на готову картинку |

**Якщо тобі просто треба згенерувати фото — відкрий тільки Fooocus (7865). Identity Studio не обов'язковий.**

Run all cells top-to-bottom.

In [ ]:
# 1) Clone both repos and install base deps
import os
import shutil
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pygit2==1.15.1"], check=True)


def rm_path(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)


def git_clone(repo_url: str, dest: str, branch: str = "main") -> None:
    rm_path(dest)
    print(f"Cloning {repo_url} -> {dest} ...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", branch, repo_url, dest],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("git stdout:", result.stdout)
        print("git stderr:", result.stderr)
        raise RuntimeError(f"git clone failed for {dest}. Check network and rerun this cell.")
    print(f"OK cloned: {dest}")


git_clone("https://github.com/sunsideaspect/foocus_new.git", "/content/Fooocus")
git_clone("https://github.com/sunsideaspect/foocus_pro.git", "/content/foocus_pro")

if not os.path.isfile("/content/Fooocus/entry_with_update.py"):
    raise RuntimeError("Fooocus clone incomplete: entry_with_update.py missing")
if not os.path.isdir("/content/foocus_pro/colab"):
    raise RuntimeError("foocus_pro clone incomplete: colab/ missing")

os.chdir("/content/foocus_pro")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "colab/requirements.txt"], check=True)

import torch
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| cuda_available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU required: Runtime -> Change runtime type -> GPU, then Restart runtime and Run all")
print("OK: /content/Fooocus and /content/foocus_pro ready")

In [ ]:
# 2) Launch Fooocus New UI in background and print Colab proxy URL
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

import torch
from IPython.display import HTML, display
from google.colab import output


def ensure_repo(path: str, clone_args: list[str]) -> None:
    marker = os.path.join(path, 'entry_with_update.py') if path.endswith('Fooocus') else os.path.join(path, 'colab')
    if os.path.exists(marker):
        return
    print(f'[Setup] Cloning into {path} ...')
    subprocess.run(clone_args, check=True)
    if not os.path.exists(marker):
        raise RuntimeError(f'Clone failed: {path}. Run cell 1 first.')


ensure_repo('/content/Fooocus', ['git', 'clone', '-b', 'main', 'https://github.com/sunsideaspect/foocus_new.git', '/content/Fooocus'])
ensure_repo('/content/foocus_pro', ['git', 'clone', '-b', 'main', 'https://github.com/sunsideaspect/foocus_pro.git', '/content/foocus_pro'])

os.chdir('/content/Fooocus')
PORT = 7865
HEALTH_URL = f'http://127.0.0.1:{PORT}/'
MAX_WAIT_SECONDS = 15 * 60
POLL_SECONDS = 2
log_path = '/content/fooocus.log'
preset_name = 'realistic_juggernaut_ragnarok'

total_vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024)) if torch.cuda.is_available() else 0
vram_flag = '--always-high-vram' if total_vram_mb >= 20000 else '--always-normal-vram'

cmd = [
    sys.executable,
    '-u',
    'entry_with_update.py',
    '--preset', preset_name,
    '--disable-censor',
    '--disable-pro-mode',
    '--listen', '0.0.0.0',
    '--port', str(PORT),
    vram_flag,
    '--disable-in-browser',
    '--disable-preset-selection',
]

subprocess.run(['bash', '-lc', "pkill -f 'entry_with_update.py'"], check=False)
open(log_path, 'w').close()
with open(log_path, 'a') as f:
    p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
print('Fooocus PID:', p.pid)

def local_http_ready(url: str) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return r.status in (200, 301, 302, 307, 308)
    except urllib.error.HTTPError as e:
        return e.code in (301, 302, 307, 308)
    except Exception:
        return False

proxy_url = None
start = time.time()
while time.time() - start < MAX_WAIT_SECONDS:
    if proxy_url is None:
        try:
            proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
        except Exception:
            proxy_url = None
    if local_http_ready(HEALTH_URL) and proxy_url:
        print('\n✅ Open Fooocus URL:\n' + proxy_url)
        display(HTML(f'<a href="{proxy_url}" target="_blank">Open Fooocus (Colab proxy)</a>'))
        break
    time.sleep(POLL_SECONDS)

subprocess.run(['bash', '-lc', 'tail -n 50 /content/fooocus.log'], check=False)

In [ ]:
# 3) Install identity refine stack (best effort)
import os
import subprocess
import time

import requests

os.chdir('/content/foocus_pro')
subprocess.call('bash colab/setup_max_identity.sh /content/foocus_pro', shell=True)
subprocess.call('pkill -f "uvicorn colab.arcface_similarity_server:app"', shell=True)
subprocess.call(
    'nohup python -m uvicorn colab.arcface_similarity_server:app --host 0.0.0.0 --port 8890 >/content/arcface.log 2>&1 &',
    shell=True,
)
time.sleep(2)
try:
    r = requests.get('http://127.0.0.1:8890/health', timeout=10)
    print('arcface health:', r.status_code, r.text)
except Exception as e:
    print('arcface health check failed:', e)

In [ ]:
# 4) Launch Identity Studio ULTIMATE UI and print URL
import os
import subprocess
import time

from IPython.display import HTML, display
from google.colab import output

os.chdir('/content/foocus_pro')
PORT = 7866
subprocess.run(['bash', '-lc', "pkill -f 'colab.launch_ultimate'"], check=False)
subprocess.check_call(
    "nohup python -m colab.launch_ultimate --port 7866 >/content/ultimate.log 2>&1 &",
    shell=True,
)
time.sleep(3)
ultimate_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
print('\n✅ Open Identity Studio URL:\n' + str(ultimate_url))
display(HTML(f'<a href="{ultimate_url}" target="_blank">Open Identity Studio ULTIMATE</a>'))
!tail -n 40 /content/ultimate.log

## How to use

1. In **Fooocus UI** (port 7865), generate the best base image and save/download it.
2. In **Identity Studio ULTIMATE** (port 7866):
   - upload identity reference face,
   - upload base generated image from Fooocus,
   - add prompt notes,
   - click `Generate ULTIMATE Photo` for face lock refine.